# RAG application running locally on Intel Xeon CPU using langchain and open-source models

Author - Pratool Bharti (pratool.bharti@intel.com)

In this cookbook, we use langchain tools and open source models to execute locally on CPU. This notebook has been validated to run on Intel Xeon 8480+ CPU. Here we implement a RAG pipeline for Llama2 model to answer questions about Intel Q1 2024 earnings release.

**Create a conda or virtualenv environment with python >=3.10 and install following libraries**
<br>

`pip install --upgrade langchain langchain-community langchainhub langchain-chroma bs4 gpt4all pypdf pysqlite3-binary` <br>
`pip install llama-cpp-python   --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu`

In [1]:
! pip install --upgrade langchain langchain-community langchainhub langchain-chroma bs4 gpt4all pypdf pysqlite3-binary
! pip install llama-cpp-python   --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 137.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 101.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.3.26
    Uninstalling langchain-community-0.3.26:
      Successfully uninstalled langchain-community-0.3.26
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.0/79.0 MB 22.0 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
  Created wheel for llama-cpp-python: filename=llama_cpp_python

**Load pysqlite3 in sys modules since ChromaDB requires sqlite3.**

In [1]:
__import__("pysqlite3")
import sys

sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

**Import essential components from langchain to load and split data**

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.text import TextLoader

In [7]:
! pwd

/home/hgolawska/llm_summer_project/Dynamics-Language-Model/work/langchain/run_locally


**Download Intel Q1 2024 earnings release**

In [8]:
!wget  'https://d1io3yog0oux5.cloudfront.net/_11d435a500963f99155ee058df09f574/intel/db/887/9014/earnings_release/Q1+24_EarningsRelease_FINAL.pdf' -O /home/hgolawska/llm_summer_project/Dynamics-Language-Model/work/langchain/run_locally/intel_q1_2024_earnings.pdf

--2025-07-04 17:24:09--  https://d1io3yog0oux5.cloudfront.net/_11d435a500963f99155ee058df09f574/intel/db/887/9014/earnings_release/Q1+24_EarningsRelease_FINAL.pdf
Resolving d1io3yog0oux5.cloudfront.net (d1io3yog0oux5.cloudfront.net)... 18.165.251.124, 18.165.251.54, 18.165.251.176, ...
Connecting to d1io3yog0oux5.cloudfront.net (d1io3yog0oux5.cloudfront.net)|18.165.251.124|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 133510 (130K) [application/pdf]
Saving to: ‘/home/hgolawska/llm_summer_project/Dynamics-Language-Model/work/langchain/run_locally/intel_q1_2024_earnings.pdf’

/home/hgolawska/llm 100%[===================>] 130.38K   573KB/s    in 0.2s    

2025-07-04 17:24:10 (573 KB/s) - ‘/home/hgolawska/llm_summer_project/Dynamics-Language-Model/work/langchain/run_locally/intel_q1_2024_earnings.pdf’ saved [133510/133510]



**Loading earning release pdf document through PyPDFLoader**

In [4]:
loader = TextLoader('ig.txt')
data = loader.load()
print(data)

[Document(metadata={'source': 'ig.txt'}, page_content='The 2023 Ig Nobel Prize Winners\nThe 2023 Ig Nobel Prizes were awarded at the 33rd First Annual Ig Nobel Prize ceremony, on Thursday, September 14, 2023. The ceremony was webcast.\n\nCHEMISTRY and GEOLOGY PRIZE [POLAND, UK]\nJan Zalasiewicz, for explaining why many scientists like to lick rocks.\nREFERENCE: “Eating Fossils,” Jan Zalasiewicz, The Paleontological Association Newsletter, no. 96, November 2017.\nWHO TOOK PART IN THE CEREMONY: Jan Zalasiewicz\n\nLITERATURE PRIZE [FRANCE, UK, MALAYSIA, FINLAND]\nChris Moulin, Nicole Bell, Merita Turunen, Arina Baharin, and Akira O’Connor for studying the sensations people feel when they repeat a single word many, many, many, many, many, many, many times.\nREFERENCE: “The The The The Induction of Jamais Vu in the Laboratory: Word Alienation and Semantic Satiation,” Chris J. A. Moulin, Nicole Bell, Merita Turunen, Arina Baharin, and Akira R. O’Connor, Memory, vol. 29, no. 7, 2021, pp. 933-

**Splitting entire document in several chunks with each chunk size is 500 tokens**

In [22]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=0)
all_splits = text_splitter.split_documents(data)

**Looking at the first split of the document**

In [23]:
all_splits[0]

Document(metadata={'source': 'ig.txt'}, page_content='The 2023 Ig Nobel Prize Winners\nThe 2023 Ig Nobel Prizes were awarded at the 33rd First Annual Ig Nobel Prize ceremony, on Thursday, September 14, 2023. The ceremony was webcast.\n\nCHEMISTRY and GEOLOGY PRIZE [POLAND, UK]\nJan Zalasiewicz, for explaining why many scientists like to lick rocks.\nREFERENCE: “Eating Fossils,” Jan Zalasiewicz, The Paleontological Association Newsletter, no. 96, November 2017.\nWHO TOOK PART IN THE CEREMONY: Jan Zalasiewicz\n\nLITERATURE PRIZE [FRANCE, UK, MALAYSIA, FINLAND]\nChris Moulin, Nicole Bell, Merita Turunen, Arina Baharin, and Akira O’Connor for studying the sensations people feel when they repeat a single word many, many, many, many, many, many, many times.\nREFERENCE: “The The The The Induction of Jamais Vu in the Laboratory: Word Alienation and Semantic Satiation,” Chris J. A. Moulin, Nicole Bell, Merita Turunen, Arina Baharin, and Akira R. O’Connor, Memory, vol. 29, no. 7, 2021, pp. 933-9

**One of the major step in RAG is to convert each split of document into embeddings and store in a vector database such that searching relevant documents are efficient.** <br>
**For that, importing Chroma vector database from langchain. Also, importing open source GPT4All for embedding models**

In [24]:
from langchain_chroma import Chroma
from langchain_community.embeddings import GPT4AllEmbeddings

**In next step, we will download one of the most popular embedding model "all-MiniLM-L6-v2". Find more details of the model at this link https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2**

In [25]:
model_name = "all-MiniLM-L6-v2.gguf2.f16.gguf"
gpt4all_kwargs = {"allow_download": "True"}
embeddings = GPT4AllEmbeddings(model_name=model_name, gpt4all_kwargs=gpt4all_kwargs)

**Store all the embeddings in the Chroma database**

In [26]:
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


**Now, let's find relevant splits from the documents related to the question**

In [27]:
question = "Who won the 2014 ig nobel prize for psychology?"
docs = vectorstore.similarity_search(question)
print(len(docs))

4


**Look at the first retrieved document from the vector database**

In [28]:
docs[0]
with open("retrieved_chunks.txt", "w") as f:
    for doc in docs:
        f.write(doc.page_content + "\n\n")
        f.write("-" * 80 + "\n\n")

**Download Lllama-2 model from Huggingface and store locally** <br>
**You can download different quantization variant of Lllama-2 model from the link below. We are using Q8 version here (7.16GB).** <br>
https://huggingface.co/TheBloke/Llama-2-7B-Chat-GGUF

In [29]:
#!huggingface-cli download TheBloke/Llama-2-7b-Chat-GGUF llama-2-7b-chat.Q8_0.gguf --local-dir . --local-dir-use-symlinks False

**Import langchain components required to load downloaded LLMs model**

In [30]:
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

**Loading the local Lllama-2 model using Llama-cpp library**

In [31]:
llm = LlamaCpp(
    model_path="llama-2-7b-chat.Q8_0.gguf",
    n_gpu_layers=-1,
    n_batch=512,
    n_ctx=2048,
    f16_kv=True,  # MUST set to True, otherwise you will run into problem after a couple of calls
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from llama-2-7b-chat.Q8_0.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u32              = 32

load: token to piece cache size = 0.1684 MB
print_info: arch             = llama
print_info: vocab_only       = 0
print_info: n_ctx_train      = 4096
print_info: n_embd           = 4096
print_info: n_layer          = 32
print_info: n_head           = 32
print_info: n_head_kv        = 32
print_info: n_rot            = 128
print_info: n_swa            = 0
print_info: is_swa_any       = 0
print_info: n_embd_head_k    = 128
print_info: n_embd_head_v    = 128
print_info: n_gqa            = 1
print_info: n_embd_k_gqa     = 4096
print_info: n_embd_v_gqa     = 4096
print_info: f_norm_eps       = 0.0e+00
print_info: f_norm_rms_eps   = 1.0e-06
print_info: f_clamp_kqv      = 0.0e+00
print_info: f_max_alibi_bias = 0.0e+00
print_info: f_logit_scale    = 0.0e+00
print_info: f_attn_scale     = 0.0e+00
print_info: n_ff             = 11008
print_info: n_expert         = 0
print_info: n_expert_used    = 0
print_info: causal attn      = 1
print_info: pooling type     = 0
print_info: rope type        = 0


**Now let's ask the same question to Llama model without showing them the earnings release.**

In [32]:
llm.invoke(question)



The 2014 Ig Nobel Prize in Psychology was awarded to the researchers who conducted a study on whether or not dogs are able to recognize their owners' faces. The study found that dogs do, in fact, have the ability to recognize their owner's face and can even differentiate between photos of their owner and those of strangers.
The prize was awarded to Dr. Attila Hányedi of the Hungarian Academy of Sciences and Dr. Ádám Miklos of Eötvös Loránd University in Budapest, Hungary, for their paper "Dogs' Ability to Recognize Their Owners' Faces." The study found that dogs were able to recognize their owners' faces with an accuracy rate of 90%.
This Ig Nobel Prize was awarded to the researchers for their work on understanding the complex social dynamics between humans and dogs, and for shedding light on the cognitive abilities of man's best friend.

llama_perf_context_print:        load time =     208.74 ms
llama_perf_context_print: prompt eval time =     208.55 ms /    17 tokens (   12.27 ms per token,    81.52 tokens per second)
llama_perf_context_print:        eval time =   15653.13 ms /   204 runs   (   76.73 ms per token,    13.03 tokens per second)
llama_perf_context_print:       total time =   16124.64 ms /   221 tokens


'\n\nThe 2014 Ig Nobel Prize in Psychology was awarded to the researchers who conducted a study on whether or not dogs are able to recognize their owners\' faces. The study found that dogs do, in fact, have the ability to recognize their owner\'s face and can even differentiate between photos of their owner and those of strangers.\nThe prize was awarded to Dr. Attila Hányedi of the Hungarian Academy of Sciences and Dr. Ádám Miklos of Eötvös Loránd University in Budapest, Hungary, for their paper "Dogs\' Ability to Recognize Their Owners\' Faces." The study found that dogs were able to recognize their owners\' faces with an accuracy rate of 90%.\nThis Ig Nobel Prize was awarded to the researchers for their work on understanding the complex social dynamics between humans and dogs, and for shedding light on the cognitive abilities of man\'s best friend.'

**As you can see, model is giving wrong information. Correct asnwer is CCG revenue in Q1 2024 is $7.5B. Now let's apply RAG using the earning release document**

**in RAG, we modify the input prompt by adding relevent documents with the question. Here, we use one of the popular RAG prompt**

In [33]:
from langchain import hub

rag_prompt = hub.pull("rlm/rag-prompt")
rag_prompt.messages

/home/hgolawska/shproject/env/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

**Appending all retreived documents in a single document**

In [34]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

**The last step is to create a chain using langchain tool that will create an e2e pipeline. It will take question and context as an input.**

In [35]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnablePick

# Chain
chain = (
    RunnablePassthrough.assign(context=RunnablePick("context") | format_docs)
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [36]:
chain.invoke({"context": docs, "question": question})

ValueError: Requested tokens (3973) exceed context window of 2048

**Now we see the results are correct as it is mentioned in earnings release.** <br>
**To further automate, we will create a chain that will take input as question and retriever so that we don't need to retrieve documents separately**

In [ ]:
retriever = vectorstore.as_retriever()
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

**Now we only need to pass the question to the chain and it will fetch the contexts directly from the vector database to generate the answer**
<br>
**Let's try with another question**

In [ ]:
qa_chain.invoke("What kind of price did Brian Wansink win?")

Llama.generate: 63 prefix-match hit, remaining 640 prompt tokens to eval


 Brian Wansink did not win any price according to the provided context.

llama_perf_context_print:        load time =   20509.60 ms
llama_perf_context_print: prompt eval time =   42308.59 ms /   640 tokens (   66.11 ms per token,    15.13 tokens per second)
llama_perf_context_print:        eval time =    1239.98 ms /    15 runs   (   82.67 ms per token,    12.10 tokens per second)
llama_perf_context_print:       total time =   43568.12 ms /   655 tokens


' Brian Wansink did not win any price according to the provided context.'